### This notebook demonstrates how to build and evaluate a neural network regression model using TensorFlow and Keras. We will follow these steps:
1. Load and preprocess the data
2. Build the neural network model
3. Train the model
4. Evaluate the model's performance
5. Perform cross-validation and hyperparameter tuning

### 1. Load and prepare the dataset ###

In [1]:
### Import libraries ###
import pandas as pd
import numpy as np
import helper_func as hf
import os

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from torch import nn, optim
import time
import random
import torch.nn.functional as F

In [ ]:
### define data path and loader ###
data_path = './processed_data/Cleaned_VLE_Data.csv'

# Path to save the model checkpoints and parameters
nn_checkpoint_path = './nn_checkpoints'
os.makedirs(nn_checkpoint_path, exist_ok=True)

model_path = f'/nn_model_params'
os.makedirs(model_path, exist_ok=True)

def get_data_loader(data_path: str,
                    batch_size: int = 32,
                    test_size: float = 0.1,
                    val_size: float = 0.1,
                    random_state=42):
    """
    Load and preprocess the dataset, then create DataLoaders for training, validation, and testing.
    :param data_path:
    :param batch_size:
    :param test_size:
    :param val_size:
    :param random_state:
    :return: data loaders for training, validation, and testing
    """
    # 1. Read the data and preprocess
    data = pd.read_csv(data_path)
    features = [col for col in data.columns if col != 'Mole fraction']
    target = 'Mole fraction'
    data = data.dropna(subset=features)
    X = data[features].drop(columns=['Component 1', 'Component 2', 'Smiles 1', 'Smiles 2', 'mol1', 'mol2'])
    X = hf.convert_to_numeric(X)
    y = data[target]

    # 2. Convert to PyTorch tensors
    X_tensor = torch.tensor(X.values, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

    # 3. Create a TensorDataset and split into train/val/test
    dataset = TensorDataset(X_tensor, y_tensor)
    total_size = len(dataset)
    test_size = int(test_size * total_size)
    val_size = int(val_size * total_size)
    train_size = total_size - test_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(random_state))

    # 4. Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    return train_loader, val_loader, test_loader